# 04 — Join Data and Build Indicators

Merge the population data to municipality polygons, count services, compute distances, and create the vulnerability index.

Before running this notebook, set the municipality code/name columns in `scripts/config.py`.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import geopandas as gpd
import numpy as np

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from scripts.config import (
    RAW_DIR, PROCESSED_DIR, OUTPUT_DIR,
    BOUNDARIES_WFS, HEALTH_WFS, SOCIAL_WFS, POPULATION_CSV,
    METRIC_CRS, MAP_CRS, GEOGRAPHIC_CRS,
    BOUNDARY_LAYER, HEALTH_LAYER, SOCIAL_LAYER,
    MUNICIPALITY_CODE_COL, MUNICIPALITY_NAME_COL, TOTAL_POP_COL,
)
from scripts.data_sources import SOURCES
from scripts.wfs_utils import discover_wfs_layers, load_wfs_layer, download_csv, save_geodataframe, save_dataframe
from scripts.population_utils import normalize_columns, build_population_65_plus
from scripts.analysis_utils import standardize_geodataframes, build_vulnerability_index, spatial_autocorrelation, top_ranked
from scripts.plotting_utils import save_choropleth
from scripts.export_utils import export_geodataframe, export_dataframe

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
if None in (MUNICIPALITY_CODE_COL, MUNICIPALITY_NAME_COL):
    raise ValueError(
        "Set MUNICIPALITY_CODE_COL and MUNICIPALITY_NAME_COL in scripts/config.py after inspecting the boundary columns."
    )

In [ ]:
boundaries = gpd.read_file(PROCESSED_DIR / "boundaries_clean.gpkg", layer="boundaries")
health = gpd.read_file(PROCESSED_DIR / "health_clean.gpkg", layer="health")
social = gpd.read_file(PROCESSED_DIR / "social_clean.gpkg", layer="social")
pop_65 = pd.read_csv(PROCESSED_DIR / "population_65_plus.csv")

In [ ]:
boundaries[MUNICIPALITY_CODE_COL] = boundaries[MUNICIPALITY_CODE_COL].astype(str).str.strip()
if MUNICIPALITY_NAME_COL in boundaries.columns:
    boundaries[MUNICIPALITY_NAME_COL] = boundaries[MUNICIPALITY_NAME_COL].astype(str).str.strip()

pop_65["mun_code"] = pop_65["mun_code"].astype(str).str.strip()

gdf = boundaries.merge(pop_65, left_on=MUNICIPALITY_CODE_COL, right_on="mun_code", how="left")
gdf["pop_65_plus"] = gdf["pop_65_plus"].fillna(0)
gdf.head()

In [ ]:
health_join = gpd.sjoin(health, gdf[[MUNICIPALITY_CODE_COL, "geometry"]], how="left", predicate="within")
social_join = gpd.sjoin(social, gdf[[MUNICIPALITY_CODE_COL, "geometry"]], how="left", predicate="within")

health_counts = health_join.groupby(MUNICIPALITY_CODE_COL).size().reset_index(name="health_count")
social_counts = social_join.groupby(MUNICIPALITY_CODE_COL).size().reset_index(name="social_count")

gdf = gdf.merge(health_counts, on=MUNICIPALITY_CODE_COL, how="left")
gdf = gdf.merge(social_counts, on=MUNICIPALITY_CODE_COL, how="left")

gdf["health_count"] = gdf["health_count"].fillna(0)
gdf["social_count"] = gdf["social_count"].fillna(0)
gdf[["health_count", "social_count"]].describe()

In [ ]:
gdf["rep_point"] = gdf.geometry.representative_point()

health_union = health.geometry.unary_union
social_union = social.geometry.unary_union

gdf["dist_health_km"] = gdf["rep_point"].distance(health_union) / 1000.0
gdf["dist_social_km"] = gdf["rep_point"].distance(social_union) / 1000.0

In [ ]:
if TOTAL_POP_COL and TOTAL_POP_COL in gdf.columns:
    gdf["aging_ratio"] = gdf["pop_65_plus"] / gdf[TOTAL_POP_COL]
else:
    gdf["aging_ratio"] = gdf["pop_65_plus"]

gdf = build_vulnerability_index(
    gdf,
    aging_col="aging_ratio",
    dist_health_col="dist_health_km",
    dist_social_col="dist_social_km",
    service_count_col="health_count",
)
gdf[["vulnerability_index", "aging_ratio"]].head()

In [ ]:
export_geodataframe(gdf, PROCESSED_DIR / "municipality_vulnerability.gpkg")
export_dataframe(
    gdf.drop(columns="geometry"),
    PROCESSED_DIR / "municipality_vulnerability_attributes.csv"
)
print("Saved the joined and indexed dataset.")